## Import Library

In [20]:
import pandas as pd
import psycopg2
from psycopg2 import sql
import numpy as np
import psycopg2.extras as extras

## Configuration

cara config agar credential tidak ke publish 

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

CSV_FILE     = os.getenv('CSV_FILE')
DB_SCHEMA    = os.getenv('DB_SCHEMA')
DB_NAME      = os.getenv('DB_NAME')
DB_USER      = os.getenv('DB_USER')
DB_PASSWORD  = os.getenv('DB_PASSWORD')
DB_HOST      = os.getenv('DB_HOST')
DB_PORT      = os.getenv('DB_PORT')
TARGET_TABLE = os.getenv('TARGET_TABLE')

## DB CONNECTION


In [23]:
# Establishing connection to the PostgreSQL database
try:
    conn = psycopg2.connect(
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
        host=DB_HOST,
        port=DB_PORT
    )

    # Set schema setelah koneksi berhasil
    with conn.cursor() as cur:
        cur.execute(f"SET search_path TO {DB_SCHEMA}")

    print(f"Successfully Connected to PostgreSQL database: {DB_NAME}, schema: {DB_SCHEMA}")

except psycopg2.Error as e:
        print(f"Error connecting to PostgreSQL database: {e}")
        print("Please check your database credentials and connection settings.")

Successfully Connected to PostgreSQL database: SalesDB, schema: staging


## CREATE TABLE

In [24]:
# create the target table structure in postgresql
create_table_query = sql.SQL("""
        CREATE TABLE IF NOT EXISTS {} (
            customer_id SERIAL PRIMARY KEY,
            gender VARCHAR(255),
            age INTEGER,
            first_day DATE,
            annual_income NUMERIC(10, 2),
            spending_score INTEGER
        );
    """).format(sql.Identifier(TARGET_TABLE))

try:
    with conn.cursor() as cursor:
        cursor.execute(create_table_query)
        conn.commit()
        print(f"Table '{TARGET_TABLE}' already exists or has been successfully created")
except psycopg2.Error as e:
    print(f"Error creating table '{TARGET_TABLE}': {e}")
    conn.rollback()

Table 'customer' already exists or has been successfully created


## EXTRACT DATA

In [28]:
# Extract Data (E)
print("---Step 1: Extraction (E) ---")

try:
    # Read the CSV file into a Pandas DataFrame
    df = pd.read_csv(CSV_FILE)
    print(f"Data extracted successfully from '{CSV_FILE}'.")
    print(f"Initial DataFrame shape: {df.shape}")
    print("\nInitial Data Head:")
    display(df.head())
except FileNotFoundError:
    print(f" Error: The file '{CSV_FILE}' was not found. Make sure it's in the same directory.")
    df = None
except Exception as e:
    print(f" Error during CSV read: {e}")
    df = None

if df is None:
    raise SystemExit("Exiting ETL process due to extraction failure.")

---Step 1: Extraction (E) ---
Data extracted successfully from 'C:\Mini Proyek\ETL_Python\data\dataset_customer_practice.csv'.
Initial DataFrame shape: (200, 6)

Initial Data Head:


,Customer_Id,Gender,Age,First Date,Annual Income (k$),Spending Score (1-100)
0,1,Male,19,1/19/2023,15,39
1,2,Male,21,1/20/2023,15,81
2,3,Female,20,1/21/2023,16,6
3,4,Female,23,1/22/2023,16,77
4,5,Female,31,1/23/2023,17,40


In [29]:
df.dtypes

Customer_Id               int64
Gender                      str
Age                       int64
First Date                  str
Annual Income (k$)        int64
Spending Score (1-100)    int64
dtype: object

## TRANSFORM 

In [30]:
# RESET ke nama kolom original
df.columns = ['CustomerID', 'Gender', 'Age', 'First Date', 'Annual Income (k$)', 'Spending Score (1-100)']

print(f"Reset Columns: {df.columns.tolist()}")

Reset Columns: ['CustomerID', 'Gender', 'Age', 'First Date', 'Annual Income (k$)', 'Spending Score (1-100)']


In [31]:
df.columns = (df.columns
    .str.strip()
    .str.replace(r'(?<=[a-z])(?=[A-Z])', '_', regex=True)  # ✅ CamelCase → underscore
    .str.lower()                                             # lowercase
    .str.replace(r'\s+', '_', regex=True)                  # spasi → underscore
    .str.replace(r'[^a-z0-9_]', '', regex=True))           # hapus karakter spesial

df.rename(columns={
    'first_date': 'first_day',
    'annual_income_k': 'annual_income',
    'spending_score_1100': 'spending_score'
}, inplace=True)

print(f"New Columns: {df.columns.tolist()}")
# ['customer_id', 'gender', 'age', 'first_day', 'annual_income', 'spending_score']

New Columns: ['customer_id', 'gender', 'age', 'first_day', 'annual_income', 'spending_score']


In [32]:
# T2: Data Type Conversion
# Convert First Date to datetime objects with explicit format
df['first_day'] = pd.to_datetime(df['first_day'], format='%m/%d/%Y', errors='coerce')

# Ensure Annual Income is numeric
df['annual_income'] = pd.to_numeric(df['annual_income'], errors='coerce')

print("Data types adjusted (First Date to datetime, Annual Income to numeric).")

# Verify the conversion
print("\nData types after conversion:")
print(df.dtypes)
print("\nFirst few dates:")
print(df['first_day'].head())

Data types adjusted (First Date to datetime, Annual Income to numeric).

Data types after conversion:
customer_id                int64
gender                       str
age                        int64
first_day         datetime64[us]
annual_income              int64
spending_score             int64
dtype: object

First few dates:
0   2023-01-19
1   2023-01-20
2   2023-01-21
3   2023-01-22
4   2023-01-23
Name: first_day, dtype: datetime64[us]


## LOAD DATA

In [33]:
# Select relevant columns for loading
df_load = df[['customer_id', 'gender', 'age', 'first_day', 'annual_income', 'spending_score']]

In [34]:
data_tuples = [tuple(row) for row in df_load.values]

# Mengubah setiap baris DataFrame menjadi tuple, karena `psycopg2` membutuhkan format ini untuk `execute_batch`. Contoh hasilnya: [(1, 'Laptop', 'Electronics', 1200, ...), (2, 'Chair', 'Furniture', 300, ...), ...]

In [35]:
# Define the INSERT query structure
cols = ', '.join(df_load.columns)
placeholders = ', '.join(['%s'] * len(df_load.columns))

# cols → string nama kolom: "order_id, product, category, ..."
# placeholders → "%s, %s, %s, ..." sebanyak jumlah kolom, sebagai placeholder nilai yang akan diisi psycopg2 secara aman (mencegah SQL injection)

In [36]:
# Define columns to update (all except the unique key 'customer_id')
update_cols = [col for col in df_load.columns if col not in ['customer_id']]

# Mengambil semua kolom kecuali customer_id (primary key/unique key). Kolom inilah yang akan di-update jika data sudah ada.

In [37]:
set_clause = ', '.join([f"{col} = EXCLUDED.{col}" for col in update_cols])

In [38]:
insert_query = sql.SQL(f"""
    INSERT INTO {TARGET_TABLE} ({cols})
    VALUES ({placeholders})
    ON CONFLICT (customer_id)
    DO UPDATE SET {set_clause}
""")

In [39]:
try:
    with conn.cursor() as cur:
        # Use execute_batch for efficient bulk insertion
        extras.execute_batch(cur, insert_query, data_tuples)
        conn.commit()
        print(f"Successfully attempted to load {len(data_tuples)} records into '{TARGET_TABLE}'. Conflicting records were updated.")

except psycopg2.Error as e:
    print(f"Error during data loading: {e}")
    conn.rollback()

finally:
    if conn:
        conn.close()
        print(" Database connection closed.")

print("\n--- ETL Process Complete ---")

Successfully attempted to load 200 records into 'customer'. Conflicting records were updated.
 Database connection closed.

--- ETL Process Complete ---
